# Intercase-Camargo pipeline (suffix)

LS-ICE-style inter-case ("load state") features added to Camargo's suffix-generation architecture, scored by real validation-period CC MAE (exhaustive 4-config grid for real-life, hyperopt search for synthetic). Real-life uses the `ssd` trim, synthetic uses `none`.

## Real

In [ ]:
import sys
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "inter-case-camargo"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

from ts_comparison import load_splits, load_ppm_raw_predictions
from params import default_params
from trainer import InterCaseCamargoTrainer
from camargo.trainer import REPO_DIR as GLSTM_REPO
from create_prefixes_from_windows import make_three_way_split
from setttings import set_global_seed
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries

set_global_seed(1904)

REAL_DATASETS = [
    "bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
    "bpic20-dom", "bpic20-int", "helpdesk", "sepsis",
]

CAMARGO_CONFIGS = [
    {"model_type": "shared_cat_ic",   "lstm_act": "selu"},
    {"model_type": "shared_cat_ic",   "lstm_act": "tanh"},
    {"model_type": "concatenated_ic", "lstm_act": "selu"},
    {"model_type": "concatenated_ic", "lstm_act": "tanh"},
]
CAMARGO_EPOCHS = 200
TOP_N_NEXT = 5
TRIM_DIR = "intercase_real"

GLSTM_OUT = GLSTM_REPO / "output_files"
RESULTS_DIR = ROOT / "results" / "inter_case_camargo_real"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODELS = ROOT / "best_models"

print(f"{len(REAL_DATASETS)} real-life datasets: {REAL_DATASETS}")
print(f"HPO: {len(CAMARGO_CONFIGS)} exhaustive configs, epochs={CAMARGO_EPOCHS}, scored by val CC MAE")


def _to_naive(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        ts = series
    else:
        ts = pd.to_datetime(series, utc=True, format="mixed")
    return ts.dt.tz_convert(None) if ts.dt.tz is not None else ts


def _score_event_log(event_log: pd.DataFrame, cc_actual: pd.Series, tt_actual: pd.Series) -> tuple:
    """caseid/end_timestamp event log -> (cc_mae, tt_mae) against a real series."""
    el = event_log.copy()
    el["end_timestamp"] = _to_naive(el["end_timestamp"])
    pred_cc = create_concurrent_cases_timeseries(
        el, time_col="end_timestamp", case_col="caseid", window="days", plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        el, time_col="end_timestamp", case_col="caseid", window="days", plot=False)
    cc_p = pred_cc.reindex(cc_actual.index).ffill().bfill().fillna(0).to_numpy()
    tt_p = pred_tt.reindex(tt_actual.index).ffill().bfill().fillna(0).to_numpy()
    cc_mae = mean_absolute_error(cc_actual.to_numpy(), cc_p)
    tt_mae = mean_absolute_error(tt_actual.to_numpy(), tt_p)
    return round(cc_mae, 3), round(tt_mae, 3)


def make_half_prefix_test_df(df, test_df):
    """Truncates each test case to the first ceil(N/2) of its own events."""
    test_cids = set(test_df["caseid"].astype(str))
    df_full = (df[df["caseid"].astype(str).isin(test_cids)]
               .copy().sort_values(["caseid", "end_timestamp"]))
    parts = [grp.iloc[: max(1, math.ceil(len(grp) / 2))] for _, grp in df_full.groupby("caseid", sort=False)]
    return pd.concat(parts, ignore_index=True)


def _setup_camargo_regime_dir(out_dir: Path, regime_name: str) -> Path:
    """Symlinks the trained model's artifacts into a <regime_name>/ subdir."""
    regime_dir = out_dir / regime_name
    regime_params_dir = regime_dir / "parameters"
    regime_params_dir.mkdir(parents=True, exist_ok=True)
    src_params_dir = out_dir / "parameters"
    for fname in ("model_parameters.json", "resource_map.csv"):
        src = (src_params_dir / fname).resolve()
        dst = regime_params_dir / fname
        if src.exists() and not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                shutil.copy2(src, dst)
    for h5 in out_dir.glob("*.h5"):
        dst = regime_dir / h5.name
        if not dst.exists():
            try:
                dst.symlink_to(h5.resolve())
            except OSError:
                shutil.copy2(h5, dst)
    src_ic_meta = out_dir / "ic_meta.json"
    dst_ic_meta = regime_dir / "ic_meta.json"
    if src_ic_meta.exists() and not dst_ic_meta.exists():
        try:
            dst_ic_meta.symlink_to(src_ic_meta.resolve())
        except OSError:
            shutil.copy2(src_ic_meta, dst_ic_meta)
    return regime_dir

In [ ]:
def _score_one_config(name, cfg_idx, cfg, train_df, val_df, val_as_test_df, full_df, cc, tt, hpo_dir, train_split):
    """Trains (or reloads) one (model_type, lstm_act) config, scored by val CC MAE."""
    trial_id = f"cfg_{cfg_idx:02d}_{cfg['model_type']}_{cfg['lstm_act']}"
    trial_run_name = f"{name}_test_full_hpo_{trial_id}"
    trial_dir = hpo_dir / trial_id
    score_path = trial_dir / "ic_hpo_score.json"
    event_log_path = trial_dir / "event_log.csv"
    if score_path.exists():
        return json.loads(score_path.read_text())

    params = default_params(trial_run_name, max_eval=1, epochs=CAMARGO_EPOCHS)
    params["model_type"] = [cfg["model_type"]]
    params["lstm_act"] = [cfg["lstm_act"]]

    t0 = time.perf_counter()
    if list(trial_dir.glob("*.h5")):
        trainer = InterCaseCamargoTrainer.from_saved(
            train_df, val_df, val_as_test_df, trial_id, params,
            trim_dir=f"{TRIM_DIR}/{name}_hpo_trials", full_df=full_df, top_n_next=TOP_N_NEXT,
        )
        train_s = None
    else:
        trainer = InterCaseCamargoTrainer(
            train_df, val_df, val_as_test_df, trial_id, params,
            trim_dir=f"{TRIM_DIR}/{name}_hpo_trials", full_df=full_df, top_n_next=TOP_N_NEXT,
        )
        trainer.run()
        train_s = round(time.perf_counter() - t0, 1)

    if event_log_path.exists():
        event_log = pd.read_csv(event_log_path)
    else:
        gen_path = trainer.predict(train_split)
        event_log = trainer.to_event_log([gen_path])
    cc_mae, tt_mae = _score_event_log(event_log, cc["val"], tt["val"])

    trial_dir.mkdir(parents=True, exist_ok=True)
    row = {**cfg, "trial": trial_id, "val_cc_mae": cc_mae, "val_tt_mae": tt_mae, "train_s": train_s}
    score_path.write_text(json.dumps(row))
    print(f"    [{trial_id}] val CC MAE={cc_mae:.4f}  TT MAE={tt_mae:.4f}  (train {train_s}s)")
    return row


def _run_camargo_ic_hpo(name, train_df, val_df, val_as_test_df, full_df, cc, tt, hpo_dir, train_split):
    hpo_dir.mkdir(parents=True, exist_ok=True)
    print(f"  [{name}] HPO: {len(CAMARGO_CONFIGS)} exhaustive configs")
    hpo_results = []
    for i, cfg in enumerate(CAMARGO_CONFIGS):
        row = _score_one_config(name, i, cfg, train_df, val_df, val_as_test_df, full_df, cc, tt, hpo_dir, train_split)
        hpo_results.append(row)

    hpo_df = pd.DataFrame(hpo_results).sort_values("val_cc_mae")
    hpo_df.to_csv(hpo_dir.parent / "hpo_results.csv", index=False)

    best = hpo_df.iloc[0].to_dict()
    best_cfg = {"model_type": best["model_type"], "lstm_act": best["lstm_act"], "winning_trial": best["trial"]}
    (hpo_dir.parent / "best_params.json").write_text(json.dumps(best_cfg, indent=2))
    print(f"  [{name}] best config: {best_cfg['model_type']}/{best_cfg['lstm_act']}  "
          f"(val CC MAE={best['val_cc_mae']:.4f}, trial={best_cfg['winning_trial']})")
    return best_cfg

In [ ]:
def _process_dataset(name, tag="intercase"):
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name

    split = load_splits(name, "ssd", is_real=True)
    train_df, val_df, test_df, full_df = split["train"], split["val"], split["test"], split["df"]
    train_split, val_split = split["cc"]["train_split"], split["cc"]["val_split"]

    _, _, val_as_test_df = make_three_way_split(
        full_df, case_col="caseid", time_col="end_timestamp",
        train_split=train_split, val_split=train_split, full_traces=True,
    )

    hpo_dir = GLSTM_OUT / TRIM_DIR / f"{name}_hpo_trials"
    if list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}] deployed model found -> skipping HPO + deploy")
    else:
        best_cfg = _run_camargo_ic_hpo(name, train_df, val_df, val_as_test_df, full_df, split["cc"], split["tt"], hpo_dir, train_split)
        winning_trial_dir = hpo_dir / best_cfg["winning_trial"]
        out_dir.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(winning_trial_dir, out_dir)
        print(f"[{name}/{tag}] deployed {best_cfg['winning_trial']} -> {out_dir}")

    params = default_params(run_name, max_eval=1, epochs=CAMARGO_EPOCHS)
    trainer = InterCaseCamargoTrainer.from_saved(
        train_df, val_df, test_df, run_name, params,
        trim_dir=TRIM_DIR, full_df=full_df, top_n_next=TOP_N_NEXT,
    )

    event_log_path = out_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}] event_log.csv found -> skipping prediction")
        event_log = pd.read_csv(event_log_path)
    else:
        gen_path = trainer.predict(val_split)
        event_log = trainer.to_event_log([gen_path])

    cc_mae, tt_mae = _score_event_log(event_log, split["cc"]["test"], split["tt"]["test"])

    row = {
        "dataset": name, "variant": tag,
        "n_cases": int(event_log["caseid"].nunique()),
        "cc_mae": cc_mae, "tt_mae": tt_mae,
    }
    print(f"[{name}/{tag}] CC MAE={cc_mae:.3f}  TT MAE={tt_mae:.3f}  n_cases={row['n_cases']}")
    return row


def _process_dataset_half(name, tag="intercase"):
    """Half-prefix regime, prediction only -- requires _process_dataset(name) to
    have run first. Saved to its own out_dir/half_prefix/event_log.csv."""
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    if not list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}/half] skip -- run _process_dataset(\"{name}\") first")
        return None

    half_dir = _setup_camargo_regime_dir(out_dir, "half_prefix")
    event_log_path = half_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}/half] event_log.csv found -> skipping prediction")
        return pd.read_csv(event_log_path)

    split = load_splits(name, "ssd", is_real=True)
    train_df, test_df, full_df = split["train"], split["test"], split["df"]
    val_split = split["cc"]["val_split"]
    half_test_df = make_half_prefix_test_df(full_df, test_df)
    empty_val = pd.DataFrame(columns=train_df.columns)

    params = default_params(run_name, max_eval=1, epochs=CAMARGO_EPOCHS)
    trainer = InterCaseCamargoTrainer.from_saved(
        train_df, empty_val, half_test_df, "half_prefix", params,
        trim_dir=f"{TRIM_DIR}/{run_name}", full_df=full_df, top_n_next=TOP_N_NEXT,
    )
    gen_path = trainer.predict(val_split)
    event_log = trainer.to_event_log([gen_path])
    print(f"[{name}/{tag}/half] predicted {event_log['caseid'].nunique()} cases -> {event_log_path}")
    return event_log


def _process_dataset_plain(name, tag="intercase"):
    """Plain-field regime, prediction only."""
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    if not list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}/plain] skip -- run _process_dataset(\"{name}\") first")
        return None

    pf_dir = _setup_camargo_regime_dir(out_dir, "pf")
    event_log_path = pf_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}/plain] event_log.csv found -> skipping prediction")
        return pd.read_csv(event_log_path)

    from arrival import ProphetArrivalModel, compute_arrival_series
    from runner import build_sos_cases, get_inflight_cases
    from sos import (
        empirical_arrival_hour_sampler, most_frequent_first_activity, most_frequent_first_resource,
    )

    split = load_splits(name, "ssd", is_real=True)
    train_df, full_df, cc = split["train"], split["df"], split["cc"]
    val_split = cc["val_split"]

    arr_s = compute_arrival_series(full_df)
    vs = pd.Timestamp(cc["val_split"])
    if vs.tzinfo:
        vs = vs.tz_convert(None)
    am = ProphetArrivalModel()
    am.fit(arr_s[arr_s.index < vs])
    pred_arr = am.predict(cc["test"].index)

    fa = most_frequent_first_activity(train_df)
    fr = most_frequent_first_resource(train_df)
    hs = empirical_arrival_hour_sampler(train_df)
    sos_df = build_sos_cases(pred_arr, fa, fr, hs)
    inflight_df = get_inflight_cases(full_df, cc["val_split"])
    print(f"[{name}/{tag}/plain] {len(sos_df)} SOS + {inflight_df['caseid'].nunique()} in-flight cases")

    plain_test_df = pd.concat([sos_df, inflight_df], ignore_index=True)

    _real_events = full_df[["caseid", "task", "end_timestamp"]].copy()
    _real_events["end_timestamp"] = _to_naive(_real_events["end_timestamp"])
    _sos_events = sos_df[["caseid", "task", "end_timestamp"]].copy()
    _sos_events["end_timestamp"] = _to_naive(_sos_events["end_timestamp"])
    full_df_for_plain = pd.concat([_real_events, _sos_events], ignore_index=True)

    empty_val = pd.DataFrame(columns=train_df.columns)
    params = default_params(run_name, max_eval=1, epochs=CAMARGO_EPOCHS)
    trainer = InterCaseCamargoTrainer.from_saved(
        train_df, empty_val, plain_test_df, "pf", params,
        trim_dir=f"{TRIM_DIR}/{run_name}", full_df=full_df_for_plain, top_n_next=TOP_N_NEXT,
    )
    gen_path = trainer.predict(val_split)
    event_log = trainer.to_event_log([gen_path])
    print(f"[{name}/{tag}/plain] predicted {event_log['caseid'].nunique()} cases -> {event_log_path}")
    return event_log

### 1. First (full-trace)

In [ ]:
rows = []
for name in tqdm(REAL_DATASETS, desc="dataset (intercase)"):
    rows.append(_process_dataset(name))
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "raw_results.csv", index=False)
results_df = pd.DataFrame(rows)
results_df


### 2. Half-prefix

In [ ]:
for name in tqdm(REAL_DATASETS, desc="dataset (intercase, half)"):
    _process_dataset_half(name)


### 3. Plain-field

In [ ]:
for name in tqdm(REAL_DATASETS, desc="dataset (intercase, plain)"):
    _process_dataset_plain(name)


### Summary

In [ ]:
def _baseline_cc_tt(dataset, regime):
    """regime in {'full','half','plain_field'}."""
    split = load_splits(dataset, "ssd", is_real=True)
    try:
        pred_log = load_ppm_raw_predictions("camargo", regime, dataset, "ssd", True)
    except Exception as e:
        print(f"  [warn] baseline camargo/{regime} for {dataset}: {e}")
        return None, None
    if pred_log is None or pred_log.empty:
        return None, None
    return _score_event_log(pred_log, split["cc"]["test"], split["tt"]["test"])


def _dataset_summary_row(name, tag="intercase"):
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    event_log_path = out_dir / "event_log.csv"

    row = {"dataset": name}
    for regime in ["", "_half", "_plain"]:
        row[f"cc_mae__intercase{regime}"] = None
        row[f"cc_mae__baseline{regime}"] = None
        row[f"cc_mae_pct_change{regime}"] = None
        row[f"tt_mae__intercase{regime}"] = None
        row[f"tt_mae__baseline{regime}"] = None
        row[f"tt_mae_pct_change{regime}"] = None

    if not event_log_path.exists():
        return row

    split = load_splits(name, "ssd", is_real=True)
    cc_ts, tt_ts = split["cc"], split["tt"]

    regime_paths = {
        "": (out_dir / "event_log.csv", "full"),
        "_half": (out_dir / "half_prefix" / "event_log.csv", "half"),
        "_plain": (out_dir / "pf" / "event_log.csv", "plain_field"),
    }
    for suffix, (ic_path, baseline_regime) in regime_paths.items():
        if not ic_path.exists():
            continue
        ic_log = pd.read_csv(ic_path)
        cc_i, tt_i = _score_event_log(ic_log, cc_ts["test"], tt_ts["test"])
        row[f"cc_mae__intercase{suffix}"] = cc_i
        row[f"tt_mae__intercase{suffix}"] = tt_i

        cc_b, tt_b = _baseline_cc_tt(name, baseline_regime)
        if cc_b is not None:
            row[f"cc_mae__baseline{suffix}"] = cc_b
            row[f"cc_mae_pct_change{suffix}"] = round((cc_i - cc_b) / cc_b * 100, 2)
        if tt_b is not None:
            row[f"tt_mae__baseline{suffix}"] = tt_b
            row[f"tt_mae_pct_change{suffix}"] = round((tt_i - tt_b) / tt_b * 100, 2)

    return row


def print_summary_table():
    df = pd.DataFrame([_dataset_summary_row(ds) for ds in REAL_DATASETS])
    df.to_csv(RESULTS_DIR / "summary.csv", index=False)
    n_full = df["cc_mae__intercase"].notna().sum()
    n_half = df["cc_mae__intercase_half"].notna().sum()
    n_plain = df["cc_mae__intercase_plain"].notna().sum()
    print(f"\n--- summary: {n_full}/{len(df)} full complete, {n_half}/{len(df)} half complete, "
          f"{n_plain}/{len(df)} plain complete ---")
    print(df.to_string(index=False))
    return df


print_summary_table()

## Synthetic

In [ ]:
import sys
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().resolve().parent.parent  # pipelines/intercase/ -> repo root
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "inter-case-camargo"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

from ts_comparison import load_splits, load_ppm_raw_predictions
from params import default_params
from trainer import InterCaseCamargoTrainer
from camargo.trainer import REPO_DIR as GLSTM_REPO
from hpo_val_scoring import build_val_as_test_df
from setttings import set_global_seed
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries

set_global_seed(1904)

EXCLUDE_DATASETS = {"loan_recency", "o2c_recency"}
SYNTH_DATASETS = sorted(
    f.stem for f in (ROOT / "data" / "synthetic").glob("*.xes")
    if f.stem not in EXCLUDE_DATASETS
)

CAMARGO_MAX_EVAL = 10
CAMARGO_EPOCHS = 200
TOP_N_NEXT = 5
TRIM_DIR = "intercase"

GLSTM_OUT = GLSTM_REPO / "output_files"
RESULTS_DIR = ROOT / "results" / "inter_case_camargo_synthetic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{len(SYNTH_DATASETS)} synthetic datasets: {SYNTH_DATASETS}")
print(f"HPO: max_eval={CAMARGO_MAX_EVAL} epochs={CAMARGO_EPOCHS}  model_type=[shared_cat_ic, concatenated_ic]")


In [ ]:
def _process_dataset(name, tag="intercase"):
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name

    split = load_splits(name, "none", is_real=False)
    train_df, val_df, test_df, full_df = split["train"], split["val"], split["test"], split["df"]
    anchor_ts = split["cc"]["val_split"]

    params = default_params(run_name, max_eval=CAMARGO_MAX_EVAL, epochs=CAMARGO_EPOCHS)

    t0 = time.perf_counter()
    if list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}] trained model found -> loading (from_saved, no retraining)")
        trainer = InterCaseCamargoTrainer.from_saved(
            train_df, val_df, test_df, run_name, params,
            trim_dir=TRIM_DIR, full_df=full_df, top_n_next=TOP_N_NEXT,
        )
        train_s = None
    else:
        val_as_test_df = build_val_as_test_df(full_df, split["cc"]["train_split"])
        trainer = InterCaseCamargoTrainer(
            train_df, val_df, test_df, run_name, params,
            trim_dir=TRIM_DIR, full_df=full_df, top_n_next=TOP_N_NEXT,
            val_as_test_df=val_as_test_df, cc_val=split["cc"]["val"], tt_val=split["tt"]["val"],
            train_split=split["cc"]["train_split"],
        )
        trainer.run()
        train_s = round(time.perf_counter() - t0, 1)

    event_log_path = out_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}] event_log.csv found -> skipping prediction")
        event_log = pd.read_csv(event_log_path)
    else:
        gen_path = trainer.predict(anchor_ts)
        event_log = trainer.to_event_log([gen_path])

    cc_mae, tt_mae = _score_event_log(event_log, split["cc"]["test"], split["tt"]["test"])

    row = {
        "dataset": name, "variant": tag, "train_s": train_s,
        "n_cases": int(event_log["caseid"].nunique()),
        "cc_mae": cc_mae, "tt_mae": tt_mae,
    }
    print(f"[{name}/{tag}] CC MAE={cc_mae:.3f}  TT MAE={tt_mae:.3f}  n_cases={row['n_cases']}  (train {train_s}s)")
    return row


def _process_dataset_half(name, tag="intercase"):
    """Half-prefix regime, prediction only -- requires _process_dataset(name) to
    have run first. Saved to its own out_dir/half_prefix/event_log.csv."""
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    if not list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}/half] skip -- run _process_dataset(\"{name}\") first")
        return None

    half_dir = _setup_camargo_regime_dir(out_dir, "half_prefix")
    event_log_path = half_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}/half] event_log.csv found -> skipping prediction")
        return pd.read_csv(event_log_path)

    split = load_splits(name, "none", is_real=False)
    train_df, test_df, full_df = split["train"], split["test"], split["df"]
    anchor_ts = split["cc"]["val_split"]
    half_test_df = make_half_prefix_test_df(full_df, test_df)
    empty_val = pd.DataFrame(columns=train_df.columns)

    params = default_params(run_name, max_eval=CAMARGO_MAX_EVAL, epochs=CAMARGO_EPOCHS)
    trainer = InterCaseCamargoTrainer.from_saved(
        train_df, empty_val, half_test_df, "half_prefix", params,
        trim_dir=f"{TRIM_DIR}/{run_name}", full_df=full_df, top_n_next=TOP_N_NEXT,
    )
    gen_path = trainer.predict(anchor_ts)
    event_log = trainer.to_event_log([gen_path])
    print(f"[{name}/{tag}/half] predicted {event_log['caseid'].nunique()} cases -> {event_log_path}")
    return event_log


def _process_dataset_plain(name, tag="intercase"):
    """Plain-field regime, prediction only."""
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    if not list(out_dir.glob("*.h5")):
        print(f"[{name}/{tag}/plain] skip -- run _process_dataset(\"{name}\") first")
        return None

    pf_dir = _setup_camargo_regime_dir(out_dir, "pf")
    event_log_path = pf_dir / "event_log.csv"
    if event_log_path.exists():
        print(f"[{name}/{tag}/plain] event_log.csv found -> skipping prediction")
        return pd.read_csv(event_log_path)

    from arrival import ProphetArrivalModel, compute_arrival_series
    from runner import build_sos_cases, get_inflight_cases
    from sos import (
        empirical_arrival_hour_sampler, most_frequent_first_activity, most_frequent_first_resource,
    )

    split = load_splits(name, "none", is_real=False)
    train_df, full_df, cc = split["train"], split["df"], split["cc"]
    anchor_ts = cc["val_split"]

    arr_s = compute_arrival_series(full_df)
    vs = pd.Timestamp(cc["val_split"])
    if vs.tzinfo:
        vs = vs.tz_convert(None)
    am = ProphetArrivalModel()
    am.fit(arr_s[arr_s.index < vs])
    pred_arr = am.predict(cc["test"].index)

    fa = most_frequent_first_activity(train_df)
    fr = most_frequent_first_resource(train_df)
    hs = empirical_arrival_hour_sampler(train_df)
    sos_df = build_sos_cases(pred_arr, fa, fr, hs)
    inflight_df = get_inflight_cases(full_df, cc["val_split"])
    print(f"[{name}/{tag}/plain] {len(sos_df)} SOS + {inflight_df['caseid'].nunique()} in-flight cases")

    plain_test_df = pd.concat([sos_df, inflight_df], ignore_index=True)

    _real_events = full_df[["caseid", "task", "end_timestamp"]].copy()
    _real_events["end_timestamp"] = _to_naive(_real_events["end_timestamp"])
    _sos_events = sos_df[["caseid", "task", "end_timestamp"]].copy()
    _sos_events["end_timestamp"] = _to_naive(_sos_events["end_timestamp"])
    full_df_for_plain = pd.concat([_real_events, _sos_events], ignore_index=True)

    empty_val = pd.DataFrame(columns=train_df.columns)
    params = default_params(run_name, max_eval=CAMARGO_MAX_EVAL, epochs=CAMARGO_EPOCHS)
    trainer = InterCaseCamargoTrainer.from_saved(
        train_df, empty_val, plain_test_df, "pf", params,
        trim_dir=f"{TRIM_DIR}/{run_name}", full_df=full_df_for_plain, top_n_next=TOP_N_NEXT,
    )
    gen_path = trainer.predict(anchor_ts)
    event_log = trainer.to_event_log([gen_path])
    print(f"[{name}/{tag}/plain] predicted {event_log['caseid'].nunique()} cases -> {event_log_path}")
    return event_log


def _baseline_cc_tt(dataset, regime):
    """regime in {'full','half','plain_field'}."""
    split = load_splits(dataset, "none", is_real=False)
    try:
        pred_log = load_ppm_raw_predictions("camargo", regime, dataset, "none", False)
    except Exception as e:
        print(f"  [warn] baseline camargo/{regime} for {dataset}: {e}")
        return None, None
    if pred_log is None or pred_log.empty:
        return None, None
    return _score_event_log(pred_log, split["cc"]["test"], split["tt"]["test"])


def _dataset_summary_row(name, tag="intercase"):
    run_name = f"{name}_test_full"
    out_dir = GLSTM_OUT / TRIM_DIR / run_name
    event_log_path = out_dir / "event_log.csv"

    row = {"dataset": name}
    for regime in ["", "_half", "_plain"]:
        row[f"cc_mae__intercase{regime}"] = None
        row[f"cc_mae__baseline{regime}"] = None
        row[f"cc_mae_pct_change{regime}"] = None
        row[f"tt_mae__intercase{regime}"] = None
        row[f"tt_mae__baseline{regime}"] = None
        row[f"tt_mae_pct_change{regime}"] = None

    if not event_log_path.exists():
        return row

    split = load_splits(name, "none", is_real=False)
    cc_ts, tt_ts = split["cc"], split["tt"]

    regime_paths = {
        "": (out_dir / "event_log.csv", "full"),
        "_half": (out_dir / "half_prefix" / "event_log.csv", "half"),
        "_plain": (out_dir / "pf" / "event_log.csv", "plain_field"),
    }
    for suffix, (ic_path, baseline_regime) in regime_paths.items():
        if not ic_path.exists():
            continue
        ic_log = pd.read_csv(ic_path)
        cc_i, tt_i = _score_event_log(ic_log, cc_ts["test"], tt_ts["test"])
        row[f"cc_mae__intercase{suffix}"] = cc_i
        row[f"tt_mae__intercase{suffix}"] = tt_i

        cc_b, tt_b = _baseline_cc_tt(name, baseline_regime)
        if cc_b is not None:
            row[f"cc_mae__baseline{suffix}"] = cc_b
            row[f"cc_mae_pct_change{suffix}"] = round((cc_i - cc_b) / cc_b * 100, 2)
        if tt_b is not None:
            row[f"tt_mae__baseline{suffix}"] = tt_b
            row[f"tt_mae_pct_change{suffix}"] = round((tt_i - tt_b) / tt_b * 100, 2)

    return row


def print_summary_table():
    df = pd.DataFrame([_dataset_summary_row(ds) for ds in SYNTH_DATASETS])
    df.to_csv(RESULTS_DIR / "summary.csv", index=False)
    n_full = df["cc_mae__intercase"].notna().sum()
    n_half = df["cc_mae__intercase_half"].notna().sum()
    n_plain = df["cc_mae__intercase_plain"].notna().sum()
    print(f"\n--- summary: {n_full}/{len(df)} full complete, {n_half}/{len(df)} half complete, "
          f"{n_plain}/{len(df)} plain complete ---")
    print(df.to_string(index=False))
    return df


### 1. First (full-trace)

In [ ]:
rows = []
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-camargo)"):
    rows.append(_process_dataset(name))
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "raw_results.csv", index=False)
results_df = pd.DataFrame(rows)
results_df


### 2. Half-prefix

In [ ]:
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-camargo, half)"):
    _process_dataset_half(name)


### 3. Plain-field

In [ ]:
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-camargo, plain)"):
    _process_dataset_plain(name)


### Summary

In [ ]:
print_summary_table()
